In [1]:
import torch
import torch.nn as nn
import math
from datasets import load_from_disk
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

d:\ProjectLibrary\Resilient AI Challenge\Gemma-Pruning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print(torch.cuda.is_available())

True


### Load the model from huggingface

In [ ]:
""" L'utilisation de device_map="auto" permet de répartir le modèle automatiquement sur le GPU disponible."""

model_id="google/gemma-4-E4B-it"
tokenizer = AutoTokenizer.from_pretrained(model_id)
print(f"Chargement du modèle {model_id} en bfloat16...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

### Capture activations

In [ ]:
def capture_activations(model, tokenizer, dataset_path, num_samples=200):
    """
    Fait passer un échantillon de données dans le modèle et capture les activations FFN.
    """
    print(f"Chargement du dataset depuis {dataset_path}...")
    # On charge le dataset unifié mentionné dans votre README
    dataset = load_from_disk(dataset_path)
    
    # On prend un petit échantillon déterministe (pour la reproductibilité seed=42)
    sample_data = dataset["test"].shuffle(seed=42).select(range(num_samples))
    
    # Dictionnaire pour stocker les activations capturées par couche
    activations = {i: [] for i in range(len(model.model.layers))}
    
    # Définition de la fonction Hook
    def get_activation_hook(layer_idx):
        def hook(module, input, output):
            # L'entrée de down_proj est le tenseur des activations intermédiaires (les fameux neurones)
            # input est un tuple, on prend le premier élément.
            # On détache le tenseur du graphe de calcul et on le passe sur CPU pour ne pas saturer la VRAM
            act = input[0].detach().cpu()
            activations[layer_idx].append(act)
        return hook

    # On attache les hooks sur la couche 'down_proj' de chaque bloc MLP
    handles = []
    print("Mise en place des hooks sur les couches FFN...")
    for i, layer in enumerate(model.model.layers):
        handle = layer.mlp.down_proj.register_forward_hook(get_activation_hook(i))
        handles.append(handle)

    print("Passage des données dans le modèle (Inférence)...")
    model.eval()
    with torch.no_grad(): # Indispensable pour ne pas stocker les gradients (économise la VRAM)
        for item in tqdm(sample_data):
            # Adaptation basique du texte (à ajuster selon la structure exacte de votre UnifiedSample)
            # Si c'est du multimodal, il faudra passer l'image au processor ici
            text = item.get("text_prompt", item.get("question", "")) 
            inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
            inputs = {k: v.to(model.device) for k, v in inputs.items()}
            
            # Le forward pass déclenche automatiquement nos hooks
            model(**inputs)

    # Nettoyage : on retire les hooks pour ne pas polluer les futures exécutions
    for handle in handles:
        handle.remove()

    # Concaténation des activations par couche pour faciliter le calcul de l'Information Mutuelle
    for i in activations.keys():
        activations[i] = torch.cat(activations[i], dim=0)
        
    print("Capture terminée ! Les activations sont prêtes pour l'analyse.")
    return activations